# Final Deployment Model Preparation ? Frozen Random Forest

Model selection and final evaluation are already complete. Notebook 12 contains the final unbiased outer-test evaluation. This notebook does not perform evaluation or tune anything: it trains the already-frozen General-only deployment configuration on the full approved modelling dataset to create a reusable pipeline artifact for Stage 9 backend integration.

**THIS NOTEBOOK DOES NOT REPLACE OR CHANGE THE REPORTED OUTER-TEST RESULTS.**

The deployment refit includes all approved rows, including rows previously reserved for the final outer test. It therefore creates a deployment pipeline distinct from the evaluation pipeline; no new performance claim is made. Keep Notebook 12 and its saved reports as the authoritative evaluation record.

Run this notebook locally from top to bottom once. It has intentionally not been executed in Codex. The single fit may take time; the frozen resource parameters are preserved. No source files, raw data, existing notebooks, or evaluation reports are written.


## 1. Imports and project paths

Locate the repository from the working directory or its ancestors. Reuse the shared preparation and pipeline functions without duplicating learned preprocessing.


In [1]:
from pathlib import Path
import json
import platform
import sys
import time

import joblib
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted

root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (root for root in root_candidates
     if (root / 'src' / 'preprocessing.py').is_file()
     and (root / 'reports' / 'eval2' / 'frozen_final_selection.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run from the hotel-cancellation repository or a directory inside it.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    TARGET, PREDICTOR_FEATURES, NUMERICAL_FEATURES, CATEGORICAL_FEATURES,
    prepare_training_table, split_features_target,
)
from src.modeling import build_model_pipeline

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hotel_bookings.csv'
FROZEN_SELECTION_PATH = PROJECT_ROOT / 'reports' / 'eval2' / 'frozen_final_selection.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
ARTIFACT_PATH = MODELS_DIR / 'final_random_forest_pipeline.joblib'
METADATA_PATH = MODELS_DIR / 'final_model_metadata.json'
print('Project root:', PROJECT_ROOT)


Project root: d:\github\hotel-cancellation


## 2. Load and verify the frozen selection

The CSV is authoritative. Safely parse its General parameter JSON; do not replace parameters, routing, or default classification behavior. Display development stage and configuration only, without computing or copying performance metrics.


In [2]:
frozen_selection = pd.read_csv(FROZEN_SELECTION_PATH)
assert len(frozen_selection) == 1, 'Expected exactly one frozen selection row.'
frozen_row = frozen_selection.iloc[0]
assert frozen_row['selected_algorithm'] == 'Random Forest'
assert frozen_row['final_strategy'] == 'General-only'
assert frozen_row['city_route_scope'] == 'General'
assert frozen_row['resort_route_scope'] == 'General'
assert str(frozen_row['outer_test_used']).strip().lower() == 'false'

frozen_general_parameters = json.loads(frozen_row['general_parameters'])
assert isinstance(frozen_general_parameters, dict) and frozen_general_parameters
assert all(isinstance(key, str) for key in frozen_general_parameters)
display(pd.DataFrame([{
    'selected_algorithm': frozen_row['selected_algorithm'],
    'strategy': frozen_row['final_strategy'],
    'selected_development_stage': frozen_row['general_selected_stage'],
    'frozen_parameters': json.dumps(frozen_general_parameters, sort_keys=True),
}]))


,selected_algorithm,strategy,selected_development_stage,frozen_parameters
0,Random Forest,General-only,tuned,"{""bootstrap"": true, ""class_weight"": ""balanced""..."


## 3. Prepare the full approved modelling dataset

Use the existing row cleaning, exclusions, categorical normalization, and deterministic engineering. Retain the approved duplicate-looking and zero-night records; remove definite zero-guest records through the existing preparation function. No partition is created. The shared split_features_target function separates predictors from the label only.


In [3]:
raw_df = pd.read_csv(RAW_PATH)
training_table = prepare_training_table(raw_df)
assert len(training_table) == 119_210, 'Approved modelling row count changed; investigate before fitting.'
X_full, y_full = split_features_target(training_table)
assert len(PREDICTOR_FEATURES) == 30
assert len(NUMERICAL_FEATURES) == 20
assert len(CATEGORICAL_FEATURES) == 10
assert X_full.columns.is_unique
assert len(X_full.columns) == 30
assert set(X_full.columns) == set(PREDICTOR_FEATURES)
assert TARGET not in X_full.columns
X_full = X_full.loc[:, list(PREDICTOR_FEATURES)].copy()
assert len(X_full) == len(y_full) == 119_210
assert X_full.index.equals(y_full.index)
assert set(y_full.unique()) == {0, 1}
print('Full deployment training rows:', len(X_full))
print('Unencoded predictors:', X_full.shape[1])


Full deployment training rows: 119210
Unencoded predictors: 30


## 4. Build and fit the deployment pipeline exactly once

Approved preprocessing and the frozen classifier are fitted together. No separate preprocessing fit is performed. The classifier retains its native predict() decision behavior and default threshold; no risk bands or custom threshold are introduced.

**Local execution only:** the following cell is the notebook's sole training fit.


In [4]:
estimator = RandomForestClassifier(**frozen_general_parameters)
final_pipeline = build_model_pipeline(X_full, estimator)
assert list(final_pipeline.named_steps) == ['preprocessor', 'classifier']
fit_started = time.perf_counter()
final_pipeline.fit(X_full, y_full)
fit_seconds = time.perf_counter() - fit_started
print(f'Deployment fit completed in {fit_seconds:.2f} seconds.')


Deployment fit completed in 1147.41 seconds.


## 5. Verify the fitted structure

These are structural checks, not model evaluation. Step names below are taken from src/modeling.py. No predictions or performance metrics are needed.


In [5]:
assert isinstance(final_pipeline, Pipeline)
assert list(final_pipeline.named_steps) == ['preprocessor', 'classifier']
assert isinstance(final_pipeline.named_steps['preprocessor'], ColumnTransformer)
assert isinstance(final_pipeline.named_steps['classifier'], RandomForestClassifier)
check_is_fitted(final_pipeline.named_steps['preprocessor'])
check_is_fitted(final_pipeline.named_steps['classifier'])
assert final_pipeline.n_features_in_ == 30
assert final_pipeline.feature_names_in_.tolist() == list(PREDICTOR_FEATURES)
assert final_pipeline.named_steps['classifier'].classes_.tolist() == [0, 1]
assert all(
    final_pipeline.named_steps['classifier'].get_params()[key] == value
    for key, value in frozen_general_parameters.items()
)
display(pd.DataFrame([{
    'pipeline_steps': list(final_pipeline.named_steps),
    'classifier_type': type(final_pipeline.named_steps['classifier']).__name__,
    'training_rows': len(X_full),
    'predictor_count': X_full.shape[1],
}]))


,pipeline_steps,classifier_type,training_rows,predictor_count
0,"[preprocessor, classifier]",RandomForestClassifier,119210,30


## 6. Save the complete pipeline and deployment metadata

Local execution produces only models/final_random_forest_pipeline.joblib and models/final_model_metadata.json. The pipeline includes both fitted preprocessing and Random Forest. Metadata stores configuration and reproducibility information, not test metrics. A rerun replaces those same two generated outputs.

Only load joblib/pickle artifacts from trusted project sources: loading an untrusted artifact can execute code. Use compatible library versions, recorded below, in the future backend.


In [6]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, ARTIFACT_PATH, compress=3)
metadata = {
    'algorithm': str(frozen_row['selected_algorithm']),
    'strategy': str(frozen_row['final_strategy']),
    'target': TARGET,
    'positive_class': 1,
    'model_predictor_count': len(PREDICTOR_FEATURES),
    'numerical_predictor_count': len(NUMERICAL_FEATURES),
    'categorical_predictor_count': len(CATEGORICAL_FEATURES),
    'predictor_features': list(PREDICTOR_FEATURES),
    'training_rows': len(X_full),
    'frozen_parameters': frozen_general_parameters,
    'selected_development_stage': str(frozen_row['general_selected_stage']),
    'artifact_filename': ARTIFACT_PATH.name,
    'preprocessing_source': 'src/preprocessing.py',
    'pipeline_source': 'src/modeling.py',
    'frozen_selection_source': 'reports/eval2/frozen_final_selection.csv',
    'random_state': frozen_general_parameters['random_state'],
    'evaluation_results_source': 'reports/eval2/final_outer_test_results.csv',
    'evaluation_note': 'Final unbiased evaluation results come from Notebook 12; this full-data deployment refit does not replace or change them.',
    'versions': {
        'python': platform.python_version(),
        'scikit_learn': sklearn.__version__,
        'pandas': pd.__version__,
        'joblib': joblib.__version__,
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2, allow_nan=False) + '\n', encoding='utf-8')
assert ARTIFACT_PATH.is_file() and METADATA_PATH.is_file()
print('Pipeline saved:', ARTIFACT_PATH)
print('Metadata saved:', METADATA_PATH)


Pipeline saved: d:\github\hotel-cancellation\models\final_random_forest_pipeline.joblib
Metadata saved: d:\github\hotel-cancellation\models\final_model_metadata.json


## 7. Serialization smoke test only ? not model evaluation

Reload the trusted artifact once and verify the fitted structure. Do not refit, compare against labels, or compute any performance metrics. This check intentionally makes no prediction.


In [7]:
loaded_pipeline = joblib.load(ARTIFACT_PATH)
assert isinstance(loaded_pipeline, Pipeline)
assert list(loaded_pipeline.named_steps) == ['preprocessor', 'classifier']
assert isinstance(loaded_pipeline.named_steps['preprocessor'], ColumnTransformer)
assert isinstance(loaded_pipeline.named_steps['classifier'], RandomForestClassifier)
check_is_fitted(loaded_pipeline.named_steps['preprocessor'])
check_is_fitted(loaded_pipeline.named_steps['classifier'])
assert loaded_pipeline.feature_names_in_.tolist() == list(PREDICTOR_FEATURES)
assert loaded_pipeline.named_steps['classifier'].classes_.tolist() == [0, 1]
print('Serialization structure verified; no evaluation performed.')


Serialization structure verified; no evaluation performed.


## Backend Integration Contract

Stage 9 backend will:

1. Receive the 22 validated API fields defined in docs/system_input_output_contract.md (20 required fields and optional nullable country/agent).
2. Derive four arrival-date components, including the documented Sunday-start week representation and full English month name.
3. Derive the five approved engineered features using the shared source definitions.
4. Construct the exact 30-predictor DataFrame in PREDICTOR_FEATURES order, with numeric predictors and categorical string codes/normalized missing values.
5. Load models/final_random_forest_pipeline.joblib from the trusted project source.
6. Call predict() to obtain class 0 or 1, preserving default classifier behavior.
7. Call predict_proba(), locating the probability column for positive class 1 via classes_.
8. Return the documented structured response: prediction, prediction_label, cancellation_probability, model_name, and strategy.

The backend must NOT retrain the model or manually apply sklearn preprocessing outside the saved pipeline. Both hotel types use the same General model. Targets and excluded features never enter inference requests; engineered/date fields are internal, not user inputs.

After local execution, the only new artifacts are:

- models/final_random_forest_pipeline.joblib
- models/final_model_metadata.json

Final reported performance remains in Notebook 12 and reports/eval2/final_outer_test_results.csv.
